In [2]:
# Cell 1: load the FLEX tests from data/tests.json
import json, re, pandas as pd
rows = [t for t in json.load(open("data/tests.json")) if t["study_id"] == "flex" and t["outcome"]]
gas = lambda d, g: float(m.group(1)) if (m := re.search(rf"{g} ([\d.]+)%", d or "")) else 0.0
df = pd.DataFrame({
    "methanol":     [int("methanol" in (r["fuel"] or "").lower()) for r in rows],
    "oxygen_pct":   [r["oxygen_pct_min"] for r in rows],
    "pressure_atm": [r["pressure_atm_min"] for r in rows],
    "co2_pct":      [gas(r["diluent"], "CO2") for r in rows],
    "he_pct":       [gas(r["diluent"], "He") for r in rows],
    "drop_mm":      [r["droplet_diameter_mm"] for r in rows],
    "outcome":      [r["outcome"] for r in rows]}).dropna()
X, y = df.drop(columns="outcome"), df["outcome"]
print(len(df), "tests:", y.value_counts().to_dict())

261 tests: {'extinction': 179, 'disruption': 55, 'completion': 27}


In [3]:
# Cell 2: compare models honestly (5-fold cross-validation)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
cv = StratifiedKFold(5, shuffle=True, random_state=0)
models = {"baseline (always most common)": DummyClassifier(strategy="most_frequent"),
          "decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=0),
          "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          "random forest": RandomForestClassifier(200, random_state=0)}
for name, m in models.items():
    acc = cross_val_score(m, X, y, cv=cv).mean()
    bal = cross_val_score(m, X, y, cv=cv, scoring="balanced_accuracy").mean()
    print(f"{name:32s} accuracy {acc:.2f}   balanced {bal:.2f}")

baseline (always most common)    accuracy 0.69   balanced 0.33
decision tree (depth 4)          accuracy 0.77   balanced 0.54
logistic regression              accuracy 0.78   balanced 0.51
random forest                    accuracy 0.79   balanced 0.57


In [4]:
# Cell 3: look at the tree's rules, then save it for the website
from sklearn.tree import export_text
tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X, y)
print(export_text(tree, feature_names=list(X.columns)))
t = tree.tree_
json.dump({"features": list(X.columns), "classes": list(tree.classes_),
           "left": t.children_left.tolist(), "right": t.children_right.tolist(),
           "feature": t.feature.tolist(), "threshold": t.threshold.tolist(),
           "value": t.value[:, 0, :].tolist()}, open("data/fire_model.json", "w"))

|--- oxygen_pct <= 23.50
|   |--- drop_mm <= 1.65
|   |   |--- pressure_atm <= 1.01
|   |   |   |--- co2_pct <= 5.00
|   |   |   |   |--- class: completion
|   |   |   |--- co2_pct >  5.00
|   |   |   |   |--- class: disruption
|   |   |--- pressure_atm >  1.01
|   |   |   |--- class: completion
|   |--- drop_mm >  1.65
|   |   |--- oxygen_pct <= 18.50
|   |   |   |--- drop_mm <= 2.31
|   |   |   |   |--- class: extinction
|   |   |   |--- drop_mm >  2.31
|   |   |   |   |--- class: extinction
|   |   |--- oxygen_pct >  18.50
|   |   |   |--- drop_mm <= 2.40
|   |   |   |   |--- class: disruption
|   |   |   |--- drop_mm >  2.40
|   |   |   |   |--- class: extinction
|--- oxygen_pct >  23.50
|   |--- oxygen_pct <= 24.50
|   |   |--- drop_mm <= 3.39
|   |   |   |--- methanol <= 0.50
|   |   |   |   |--- class: disruption
|   |   |   |--- methanol >  0.50
|   |   |   |   |--- class: extinction
|   |   |--- drop_mm >  3.39
|   |   |   |--- class: extinction
|   |--- oxygen_pct >  24.50
| 

In [5]:
yb = (y == "extinction")
for name, m in models.items():
    print(f"{name:32s} balanced {cross_val_score(m, X, yb, cv=cv, scoring='balanced_accuracy').mean():.2f}")

baseline (always most common)    balanced 0.50
decision tree (depth 4)          balanced 0.71
logistic regression              balanced 0.78
random forest                    balanced 0.78


In [6]:
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X, yb)
sc, lr = m[0], m[-1]
json.dump({"question": "Does the flame go out on its own?", "features": list(X.columns),
           "mean": sc.mean_.tolist(), "scale": sc.scale_.tolist(),
           "weights": lr.coef_[0].tolist(), "intercept": float(lr.intercept_[0]),
           "ranges": {c: [float(X[c].min()), float(X[c].max())] for c in X.columns},
           "accuracy": 0.84, "balanced_accuracy": 0.78, "baseline": 0.69, "tests": len(X)},
          open("data/fire_model.json", "w"), indent=2)